In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the four sliders. The red and blue arrows are $z_1$ and $z_2$, and
the gold arrow is their product. Watch the two angles add and the two
lengths multiply. The gray circle marks a magnitude of 1, where
multiplying only rotates.

In [ ]:
# hide
# autorun
R1_0, A1_0, R2_0, A2_0 = 1.2, 35.0, 1.0, 55.0    # starting parameters

CIRC = np.linspace(0.0, 2 * np.pi, 361)

def vec(r, deg):
    a = np.radians(deg)
    return r * np.cos(a), r * np.sin(a)

def arc(deg, radius, n=120):
    a = np.linspace(0.0, np.radians(deg), n)
    return radius * np.cos(a), radius * np.sin(a)

def figure():
    fig = go.Figure()
    fig.add_scatter(x=np.cos(CIRC), y=np.sin(CIRC), mode="lines",
                    line=dict(color=STEEL, width=1.4, dash="dot"))
    for deg, radius, color in ((A1_0, 0.34, RED), (A2_0, 0.50, BLUE),
                               (A1_0 + A2_0, 0.66, GOLD)):
        ax, ay = arc(deg, radius)
        fig.add_scatter(x=ax, y=ay, mode="lines",
                        line=dict(color=color, width=1.4, dash="dot"))
    for r, deg, color, width in ((R1_0, A1_0, RED, 2.6),
                                 (R2_0, A2_0, BLUE, 2.6),
                                 (R1_0 * R2_0, A1_0 + A2_0, GOLD, 3.4)):
        vx, vy = vec(r, deg)
        fig.add_scatter(x=[0, vx], y=[0, vy], mode="lines+markers",
                        line=dict(color=color, width=width),
                        marker=dict(color=color, size=[1, 10]))
    # a tall figure with a narrow x domain keeps the plotting area square, so
    # the equal-aspect constraint does not stretch the frame sideways
    fig.update_layout(height=520)
    fig.update_xaxes(domain=[0.24, 0.76], range=[-2.6, 2.6], title_text="Re",
                     fixedrange=True, scaleanchor="y", scaleratio=1)
    fig.update_yaxes(range=[-2.6, 2.6], title_text="Im", fixedrange=True)
    return fig

def controls(fig):
    r1 = widgets.FloatSlider(description="r_1", min=0.2, max=1.5, value=R1_0,
                             step=0.05)
    a1 = widgets.FloatSlider(description="angle_1 (deg)", min=-180, max=180,
                             value=A1_0, step=5)
    r2 = widgets.FloatSlider(description="r_2", min=0.2, max=1.5, value=R2_0,
                             step=0.05)
    a2 = widgets.FloatSlider(description="angle_2 (deg)", min=-180, max=180,
                             value=A2_0, step=5)
    readout = widgets.HTML()

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def update(r1, a1, r2, a2, vec=vec, arc=arc, readout=readout):
        with fig.batch_update():
            for i, (deg, radius) in enumerate(((a1, 0.34), (a2, 0.50),
                                               (a1 + a2, 0.66))):
                fig.data[1 + i].x, fig.data[1 + i].y = arc(deg, radius)
            for i, (r, deg) in enumerate(((r1, a1), (r2, a2),
                                          (r1 * r2, a1 + a2))):
                vx, vy = vec(r, deg)
                fig.data[4 + i].x, fig.data[4 + i].y = [0, vx], [0, vy]
        x1, y1 = vec(r1, a1)
        x2, y2 = vec(r2, a2)
        wrapped = (a1 + a2 + 180) % 360 - 180
        zero = lambda v: 0.0 if abs(v) < 0.005 else v   # no "-0.00" readouts
        x1, y1, x2, y2 = zero(x1), zero(y1), zero(x2), zero(y2)
        px, py = zero(x1 * x2 - y1 * y2), zero(x1 * y2 + x2 * y1)
        readout.value = (
            f"<span style='font-size:0.9em'>"
            f"z_1 = {x1:+.2f} {y1:+.2f}j &nbsp;·&nbsp; "
            f"z_2 = {x2:+.2f} {y2:+.2f}j &nbsp;·&nbsp; "
            f"z_1 z_2 = {px:+.2f} {py:+.2f}j "
            f"&nbsp;=&nbsp; length {r1 * r2:.2f} at {wrapped:.0f} deg"
            f"</span>")

    widgets.interactive_output(update, {"r1": r1, "a1": a1, "r2": r2, "a2": a2})
    return widgets.VBox([r1, a1, r2, a2, readout])

icm_plotly.show(figure, controls)